# Annotator agreement

Adds the annotator agreement level (50 / 66 / 75 / 100%) to every sentence of train, val and test.
The `all_agree` flag is used in notebook 04 (targeted training on the transformer) and notebook 07 (error taxonomy and the numbers for report sections 5.4 and 5.5), so run this notebook before those two.

## Imports

In [ ]:
import os

import pandas as pd

os.makedirs("../results/hard_samples", exist_ok=True)

## Agreement level of every sentence

Financial PhraseBank comes in four files: sentences with at least 50%, 66%, 75% and 100% annotator agreement. Every sentence gets the highest level it appears in, and `all_agree` = 100%.
Some sentences have broken characters (for example "+Æ"), so we match on letters and digits only. Upper and lower case stay different, because a few sentences appear twice with different capitals and different agreement.
Saved to `results/hard_samples/agreement_{train,val,test}.csv`. Notebooks 04 and 07 use these files.

In [ ]:
import re

def match_key(sentence):
    return re.sub(r"[^A-Za-z0-9]", "", sentence)   # letters and digits only

agreement_level = {}
for level, name in [(50, "50Agree"), (66, "66Agree"), (75, "75Agree"), (100, "AllAgree")]:
    with open(f"../data/Sentences_{name}.txt", encoding="latin-1") as f:
        for line in f:
            if line.strip():
                agreement_level[match_key(line.rsplit("@", 1)[0])] = level   # higher levels overwrite lower ones

for split in ["train", "val", "test"]:
    df = pd.read_csv(f"../data/{split}.csv")
    out = df[["sentence", "label"]].copy()
    out["agreement"] = [agreement_level.get(match_key(s), 0) for s in df.sentence]
    out["all_agree"] = out.agreement == 100
    out.to_csv(f"../results/hard_samples/agreement_{split}.csv", index=False)
    print(split, "| all-agree:", out.all_agree.sum(), "/", len(out), "| not found:", (out.agreement == 0).sum())